# Surgical Phase Recognition — Inference Demo

This notebook demonstrates the full SurgPhase inference pipeline:
1. Load a pretrained model (EfficientNet-B4 spatial encoder + MS-TCN temporal model)
2. Run single-frame phase prediction
3. Apply temporal smoothing over a synthetic video sequence
4. Visualize phase transitions as a timeline chart

Phases follow the Cholec80 annotation scheme (7 phases).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

print(f"torch: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

## Load Model

The full pipeline consists of:
- **SpatialEncoder**: EfficientNet-B4 backbone, outputs 512-d frame features
- **MSTCN**: Multi-Stage Temporal Convolutional Network for phase sequence labeling

In [ ]:
from src.models.spatial_encoder import EfficientNetEncoder
from src.models.temporal_model import MSTCN
from src.models.phase_classifier import SurgPhaseClassifier

PHASE_NAMES = [
    'preparation',
    'calot triangle dissection',
    'clipping and cutting',
    'gallbladder dissection',
    'gallbladder packaging',
    'cleaning and coagulation',
    'gallbladder retraction',
]
NUM_PHASES = len(PHASE_NAMES)

# instantiate components
encoder = EfficientNetEncoder(out_features=512, pretrained=False).to(device)
classifier = SurgPhaseClassifier(
    feature_dim=512,
    num_phases=NUM_PHASES,
).to(device)

encoder.eval()
classifier.eval()
print("models instantiated")

## Single-Frame Phase Prediction

In [ ]:
# synthetic frame: (B, C, H, W)
frame = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    features = encoder(frame)          # (1, 512)
    logits = classifier(features)      # (1, NUM_PHASES)
    probs = torch.softmax(logits, dim=-1)
    pred_phase = probs.argmax(dim=-1).item()

print(f"feature shape: {features.shape}")
print(f"predicted phase: {pred_phase} — '{PHASE_NAMES[pred_phase]}'")
print(f"confidence: {probs[0, pred_phase].item():.3f}")

# top-3
top3 = probs[0].topk(3)
for score, idx in zip(top3.values, top3.indices):
    print(f"  {PHASE_NAMES[idx.item()]:35s} {score.item():.3f}")

## Temporal Smoothing Demo

Raw per-frame predictions are noisy. We apply a sliding-window majority vote to smooth
out short flickers — standard post-processing for phase recognition.

In [ ]:
def majority_vote_smooth(predictions, window=15):
    """Sliding-window majority vote smoothing."""
    smoothed = predictions.copy()
    half = window // 2
    for i in range(len(predictions)):
        lo = max(0, i - half)
        hi = min(len(predictions), i + half + 1)
        smoothed[i] = Counter(predictions[lo:hi]).most_common(1)[0][0]
    return smoothed

# synthetic sequence simulating a 10-minute cholecystectomy at 1 fps
rng = np.random.default_rng(7)
T = 600  # frames

# ground-truth phase durations (approximate Cholec80 statistics)
phase_durations = [60, 180, 40, 200, 30, 50, 40]
gt_sequence = []
for phase_id, dur in enumerate(phase_durations):
    gt_sequence.extend([phase_id] * dur)
gt_sequence = np.array(gt_sequence[:T])

# add noise: random flips to adjacent phases
noisy = gt_sequence.copy()
flip_idx = rng.choice(T, size=60, replace=False)
noisy[flip_idx] = (noisy[flip_idx] + rng.integers(1, 3, size=60)) % NUM_PHASES

smoothed = majority_vote_smooth(noisy.tolist(), window=21)

accuracy_raw = (noisy == gt_sequence).mean()
accuracy_smooth = (np.array(smoothed) == gt_sequence).mean()
print(f"raw accuracy:      {accuracy_raw:.3f}")
print(f"smoothed accuracy: {accuracy_smooth:.3f}")

## Phase Transition Visualization

In [ ]:
colors = plt.cm.Set2(np.linspace(0, 1, NUM_PHASES))

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
t = np.arange(T)

for ax, seq, title in zip(
    axes,
    [gt_sequence, noisy, np.array(smoothed)],
    ['ground truth', 'raw predictions (noisy)', 'smoothed predictions']
):
    for phase_id in range(NUM_PHASES):
        mask = seq == phase_id
        ax.fill_between(t, 0, 1, where=mask, color=colors[phase_id], alpha=0.8)
    ax.set_ylabel(title, fontsize=9)
    ax.set_yticks([])

axes[-1].set_xlabel('frame', fontsize=10)

patches = [mpatches.Patch(color=colors[i], label=PHASE_NAMES[i]) for i in range(NUM_PHASES)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Surgical Phase Timeline — Cholec80 style', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('phase_timeline.png', dpi=120, bbox_inches='tight')
plt.show()

# bar chart of predicted phase durations
phase_counts = Counter(smoothed)
fig2, ax2 = plt.subplots(figsize=(10, 4))
bars = ax2.bar(
    PHASE_NAMES,
    [phase_counts.get(i, 0) for i in range(NUM_PHASES)],
    color=colors
)
ax2.set_xlabel('phase')
ax2.set_ylabel('duration (frames)')
ax2.set_title('predicted phase durations')
plt.xticks(rotation=20, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('phase_durations.png', dpi=120, bbox_inches='tight')
plt.show()